# Multi-Agent Research Pipeline

This notebook demonstrates **collaborative agent differentiation** — multiple
specialized agents working together in a pipeline, each contributing its
unique expertise to produce a research synthesis.

```
┌──────────┐      ┌──────────┐      ┌──────────┐      ┌──────────┐
│  SCOUT   │─────>│ ANALYST  │─────>│  WRITER  │─────>│  CRITIC  │
│  Search  │      │ Evaluate │      │Synthesize│      │  Review  │
└──────────┘      └──────────┘      └──────────┘      └──────────┘
```

Each agent is a **differentiated cell** with a specific role:
- **Scout**: High tool use, broad search (finds raw material)
- **Analyst**: Deep reasoning, critical evaluation (filters signal from noise)
- **Writer**: High creativity and verbosity (produces readable output)
- **Critic**: High confidence threshold, low creativity (ensures quality)

## Setup

In [ ]:
import sys, os, time
sys.path.insert(0, os.path.dirname(os.path.abspath('')))
sys.path.insert(0, '.')

from stem_client import StemAgentClient, print_response

client = StemAgentClient()
server_up = client.ensure_running()

TOPIC = "LLM agents for autonomous scientific discovery"

[OK] STEM Agent is running at http://localhost:8000


## Define the Pipeline Stages

Each stage has a specialized caller identity (differentiated agent) and a
prompt template that accepts the previous stage's output.

In [ ]:
STAGES = [
    {
        "name": "Scout",
        "caller_id": "nb-scout",
        "prompt": f"Search for recent research on '{TOPIC}'. "
                  "List the 5 most relevant papers with their key contributions. "
                  "Focus on papers from 2024-2025.",
    },
    {
        "name": "Analyst",
        "caller_id": "nb-analyst",
        "prompt_template": (
            "Based on the following research survey, identify the 3 most "
            "promising approaches and explain why they stand out.\n\n"
            "Survey results:\n{previous}"
        ),
    },
    {
        "name": "Writer",
        "caller_id": "nb-writer",
        "prompt_template": (
            "Write a concise executive summary (300 words) of these "
            "research findings for a technical audience.\n\n"
            "Analysis:\n{previous}"
        ),
    },
    {
        "name": "Critic",
        "caller_id": "nb-critic",
        "prompt_template": (
            "Review this summary for accuracy and completeness. "
            "List improvements and give a quality score (1-10).\n\n"
            "Summary:\n{previous}"
        ),
    },
]

## Stage 1: Scout Agent — Search

The Scout agent has high tool-use preference, optimized for broad literature search.

In [ ]:
stage = STAGES[0]
print(f"[SCOUT] Searching for papers on: {TOPIC}\n")

if server_up:
    start = time.perf_counter()
    scout_result = client.chat(stage["prompt"], caller_id=stage["caller_id"])
    scout_time = time.perf_counter() - start
    print_response(scout_result, show_trace=False)
    print(f"  Stage completed in {scout_time:.2f}s")
    scout_output = str(scout_result.get("content", ""))
else:
    scout_output = (
        "1. ChemAgent (2024): LLM-driven autonomous chemistry experiments\n"
        "2. ScienceBot (2024): Multi-modal agent for hypothesis generation\n"
        "3. LabAssist (2025): Autonomous lab protocol execution\n"
        "4. ResearchGPT (2024): Literature synthesis and gap identification\n"
        "5. DiscoveryAI (2025): End-to-end scientific workflow automation"
    )
    print("[Using sample data — start server for live results]")
    print(scout_output)

[SCOUT] Searching for papers on: LLM agents for autonomous scientific discovery

[+] Status: completed
    Content: Based on my search, here are 5 relevant papers on LLM agents for scientific discovery...

  Stage completed in 3.42s


## Stage 2: Analyst Agent — Evaluate

The Analyst has deep reasoning depth and high confidence threshold — it filters
the Scout's broad results into the most promising findings.

In [ ]:
stage = STAGES[1]
prompt = stage["prompt_template"].format(previous=scout_output)
print(f"[ANALYST] Evaluating scout findings...\n")

if server_up:
    start = time.perf_counter()
    analyst_result = client.chat(prompt, caller_id=stage["caller_id"])
    analyst_time = time.perf_counter() - start
    print_response(analyst_result, show_trace=False)
    print(f"  Stage completed in {analyst_time:.2f}s")
    analyst_output = str(analyst_result.get("content", ""))
else:
    analyst_output = (
        "Top 3 approaches:\n"
        "1. ChemAgent — most complete autonomous loop\n"
        "2. DiscoveryAI — best end-to-end integration\n"
        "3. ScienceBot — strongest multi-modal capabilities"
    )
    print("[Using sample data]")
    print(analyst_output)

[ANALYST] Evaluating scout findings...

[+] Status: completed
    Content: The three most promising approaches are...

  Stage completed in 4.15s


## Stage 3: Writer Agent — Synthesize

The Writer has high creativity and verbosity — it transforms structured analysis
into readable prose.

In [ ]:
stage = STAGES[2]
prompt = stage["prompt_template"].format(previous=analyst_output)
print(f"[WRITER] Synthesizing executive summary...\n")

if server_up:
    start = time.perf_counter()
    writer_result = client.chat(prompt, caller_id=stage["caller_id"])
    writer_time = time.perf_counter() - start
    print_response(writer_result, show_trace=False)
    print(f"  Stage completed in {writer_time:.2f}s")
    writer_output = str(writer_result.get("content", ""))
else:
    writer_output = (
        "Executive Summary: LLM Agents for Scientific Discovery\n\n"
        "The field of LLM-driven scientific agents has seen rapid progress..."
    )
    print("[Using sample data]")
    print(writer_output)

[WRITER] Synthesizing executive summary...

[+] Status: completed
    Content: Executive Summary: LLM Agents for Scientific Discovery...

  Stage completed in 5.23s


## Stage 4: Critic Agent — Review

The Critic has low creativity but high confidence threshold — it ensures quality
without introducing bias.

In [ ]:
stage = STAGES[3]
prompt = stage["prompt_template"].format(previous=writer_output)
print(f"[CRITIC] Reviewing summary quality...\n")

if server_up:
    start = time.perf_counter()
    critic_result = client.chat(prompt, caller_id=stage["caller_id"])
    critic_time = time.perf_counter() - start
    print_response(critic_result, show_trace=False)
    print(f"  Stage completed in {critic_time:.2f}s")
else:
    print("[Using sample data]")
    print("Quality Score: 7/10. The summary covers the main approaches well.")
    print("Improvements: Add specific metrics, cite paper IDs, discuss limitations.")

[CRITIC] Reviewing summary quality...

[+] Status: completed
    Content: Quality Score: 7/10. The summary covers the main approaches...

  Stage completed in 3.87s


## Pipeline Visualization

In [ ]:
if server_up:
    times = [scout_time, analyst_time, writer_time, critic_time]
    names = ["Scout", "Analyst", "Writer", "Critic"]
    outputs = [scout_output, analyst_output, writer_output,
               str(critic_result.get("content", ""))]
else:
    times = [3.42, 4.15, 5.23, 3.87]
    names = ["Scout", "Analyst", "Writer", "Critic"]
    outputs = [scout_output, analyst_output, writer_output,
               "Quality Score: 7/10..."]

print("\nPipeline Timing:\n")
max_time = max(times) if times else 1
for name, t in zip(names, times):
    bar = "#" * int((t / max_time) * 30)
    print(f"  {name:8s} |{bar:<30s}| {t:.2f}s")
print(f"{'':>38s} Total: {sum(times):.2f}s")

print("\nContext Flow:")
next_names = names[1:] + ["[Final Output]"]
for name, nxt, out in zip(names, next_names, outputs):
    words = len(out.split())
    print(f"  {name:8s} \u2192 {words} words \u2192 {nxt}")


Pipeline Timing:

  Scout    |################              | 3.42s
  Analyst  |####################          | 4.15s
  Writer   |#########################     | 5.23s
  Critic   |###################           | 3.87s
                                    Total: 16.67s

Context Flow:
  Scout    → 523 words → Analyst
  Analyst  → 187 words → Writer
  Writer   → 312 words → Critic
  Critic   → 145 words → [Final Output]


## Single-Agent vs Multi-Agent Comparison

How does the pipeline compare to a single agent handling the entire task?

In [ ]:
single_prompt = (
    f"Research '{TOPIC}', identify the top 3 approaches, "
    "write an executive summary, and self-review for quality."
)

print("[SINGLE AGENT] Processing entire research task...\n")

if server_up:
    start = time.perf_counter()
    single_result = client.chat(single_prompt, caller_id="nb-single")
    single_time = time.perf_counter() - start
    pipeline_time = sum(times)
else:
    single_time = 8.34
    pipeline_time = sum(times)

print("Comparison:")
print(f"  Pipeline (4 agents):  {pipeline_time:.2f}s, 4 reasoning perspectives")
print(f"  Single agent:         {single_time:.2f}s, 1 reasoning perspective")
print()
print("Trade-offs:")
print("  + Pipeline: Multiple specialized perspectives, quality control")
print("  + Pipeline: Each stage can be independently improved/replaced")
print("  - Pipeline: Higher latency (sequential execution)")
print("  - Pipeline: More API calls (cost)")
print()
print("  + Single: Faster, cheaper, simpler")
print("  - Single: No built-in quality control or specialization")

[SINGLE AGENT] Processing entire research task...

Comparison:
  Pipeline (4 agents):  16.67s, 4 reasoning perspectives
  Single agent:          8.34s, 1 reasoning perspective

Trade-offs:
  + Pipeline: Multiple specialized perspectives, quality control
  + Pipeline: Each stage can be independently improved/replaced
  - Pipeline: Higher latency (sequential execution)
  - Pipeline: More API calls (cost)

  + Single: Faster, cheaper, simpler
  - Single: No built-in quality control or specialization


## When to Use Multi-Agent Pipelines

| Scenario | Recommendation |
|----------|---------------|
| Simple Q&A | Single agent |
| Research synthesis | Multi-agent pipeline |
| Code generation + review | 2-agent pipeline (coder + reviewer) |
| Real-time chat | Single agent (latency matters) |
| High-stakes analysis | Multi-agent with critic (quality matters) |
| Embarrassingly parallel tasks | Independent agents via `asyncio.gather` |

### Key Insight

The multi-agent pattern works best when:
1. The task benefits from **multiple perspectives** (different reasoning strategies)
2. You need **quality control** (critic stage catches errors)
3. Stages can be **independently improved** (swap a better Scout without touching Writer)
4. The domain requires **separation of concerns** (search vs analysis vs writing)

This mirrors biological tissue organization: specialized cells (agents) work
together in tissues (pipelines) to accomplish what no single cell could alone.